# Notebook 3: Feature Engineering and Model Preparation

## Purpose

This notebook converts the validated analytical and supervised panels created in Notebook 2 into a modelling-ready dataset.

The work will preserve the time order of the data. Every predictor used to forecast a next-year food-supply shortage must come from the current year or earlier years. Information from the future will not be used.

The notebook will:

1. reload and validate the saved Notebook 2 outputs;
2. identify the variables that are safe and meaningful for prediction;
3. create historical and trend-based features;
4. preserve missingness and data-quality information appropriately;
5. apply the previously agreed temporal train, validation and test periods;
6. produce a validated modelling dataset for the next stage.

No statistical model will be trained until the modelling dataset has been fully inspected and validated.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Display settings
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

# Project directories
PROJECT_DIR = Path.home() / "Documents" / "food_security_predictor"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "africa_first"

# Validated outputs from Notebook 2
ANALYTICAL_PANEL_PATH = (
    PROCESSED_DIR / "africa_analytical_panel_2010_2023.parquet"
)

SUPERVISED_PANEL_PATH = (
    PROCESSED_DIR / "africa_supervised_shortage_panel_2010_2022.parquet"
)

TARGET_DEFINITION_PATH = (
    PROCESSED_DIR / "shortage_target_definition.csv"
)

TEMPORAL_SPLIT_PATH = (
    PROCESSED_DIR / "shortage_temporal_split_summary.csv"
)

required_files = [
    ANALYTICAL_PANEL_PATH,
    SUPERVISED_PANEL_PATH,
    TARGET_DEFINITION_PATH,
    TEMPORAL_SPLIT_PATH,
]

missing_files = [
    path.name
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following validated Notebook 2 files could not be found:\n"
        + "\n".join(f"- {name}" for name in missing_files)
    )

print("Notebook 3 environment prepared successfully.")
print(f"Project directory: {PROJECT_DIR}")
print(f"Processed-data directory: {PROCESSED_DIR}")
print("\nAll required Notebook 2 files were found:")
for path in required_files:
    print(f"- {path.name}")

Notebook 3 environment prepared successfully.
Project directory: /Users/adewale/Documents/food_security_predictor
Processed-data directory: /Users/adewale/Documents/food_security_predictor/data/processed/africa_first

All required Notebook 2 files were found:
- africa_analytical_panel_2010_2023.parquet
- africa_supervised_shortage_panel_2010_2022.parquet
- shortage_target_definition.csv
- shortage_temporal_split_summary.csv


In [2]:
# Load the validated Notebook 2 outputs

analytical_panel = pd.read_parquet(ANALYTICAL_PANEL_PATH)
supervised_panel = pd.read_parquet(SUPERVISED_PANEL_PATH)
target_definition = pd.read_csv(TARGET_DEFINITION_PATH)
temporal_split_summary = pd.read_csv(TEMPORAL_SPLIT_PATH)

loaded_summary = pd.DataFrame(
    {
        "Dataset": [
            "Complete analytical panel",
            "Supervised shortage panel",
            "Target definition",
            "Temporal split summary",
        ],
        "Rows": [
            len(analytical_panel),
            len(supervised_panel),
            len(target_definition),
            len(temporal_split_summary),
        ],
        "Columns": [
            analytical_panel.shape[1],
            supervised_panel.shape[1],
            target_definition.shape[1],
            temporal_split_summary.shape[1],
        ],
    }
)

print("Loaded Notebook 2 outputs:")
display(loaded_summary)

print("\nComplete analytical-panel period:")
print(
    f"{analytical_panel['Year'].min()} to "
    f"{analytical_panel['Year'].max()}"
)

print("\nSupervised-panel predictor period:")
print(
    f"{supervised_panel['Year'].min()} to "
    f"{supervised_panel['Year'].max()}"
)

print("\nSupervised-panel identity coverage:")
identity_summary = pd.Series(
    {
        "Countries": supervised_panel["Area"].nunique(),
        "Commodities": supervised_panel["Item"].nunique(),
        "Country-commodity pairs": (
            supervised_panel[["Area", "Item"]]
            .drop_duplicates()
            .shape[0]
        ),
    },
    name="Result",
).to_frame()

display(identity_summary)

# Identify columns connected to the target, eligibility and split
search_words = [
    "target",
    "shortage",
    "eligible",
    "split",
    "next",
]

target_related_columns = [
    column
    for column in supervised_panel.columns
    if any(
        word in column.lower()
        for word in search_words
    )
]

print("\nTarget-related columns found in the supervised panel:")
for column in target_related_columns:
    print(f"- {column}")

print("\nSaved target definition:")
display(target_definition)

print("\nSaved temporal split summary:")
display(temporal_split_summary)

print("\nFirst five supervised-panel identities:")
display(
    supervised_panel[
        ["Area", "Item Code", "Item", "Year"]
    ].head()
)

Loaded Notebook 2 outputs:


,Dataset,Rows,Columns
0,Complete analytical panel,4816,84
1,Supervised shortage panel,3248,87
2,Target definition,1,15
3,Temporal split summary,3,13



Complete analytical-panel period:
2010 to 2023

Supervised-panel predictor period:
2010 to 2022

Supervised-panel identity coverage:


,Result
Countries,43
Commodities,8
Country-commodity pairs,260



Target-related columns found in the supervised panel:
- target_year
- Temporal split
- shortage_next_year

Saved target definition:


,Target column,Prediction horizon,Predictor row year,Outcome year,Minimum current kcal/capita/day,Minimum percentage decline,Minimum absolute decline kcal/capita/day,Positive definition,Negative definition,Ineligible treatment,Training target years,Validation target years,Test target years,Future relevance used as feature,Next-year outcome values included as features
0,shortage_next_year,One year ahead,Year t,Year t + 1,5,15,5,Next-year food supply kcal/capita/day falls by...,Eligible transition that does not satisfy both...,Target left missing; row excluded from supervi...,2011–2019,2020–2021,2022–2023,False,False



Saved temporal split summary:


,Temporal split,Predictor_year_start,Predictor_year_end,Target_year_start,Target_year_end,Eligible_rows,Positive_events,Negative_events,Event rate %,Non-event to event ratio,Countries,Commodities,Country_commodity_pairs
0,Train,2010,2018,"2,011.00","2,019.00",2250,231,2019,10.27,8.74,43,8,259
1,Validation,2019,2020,"2,020.00","2,021.00",500,52,448,10.40,8.62,43,8,254
2,Test,2021,2022,"2,022.00","2,023.00",498,51,447,10.24,8.76,43,8,255



First five supervised-panel identities:


,Area,Item Code,Item,Year
0,Algeria,2511,Wheat and products,2010
1,Algeria,2511,Wheat and products,2011
2,Algeria,2511,Wheat and products,2012
3,Algeria,2511,Wheat and products,2013
4,Algeria,2511,Wheat and products,2014


In [3]:
# Identify the different groups of columns in the supervised panel

identity_columns = [
    "Area Code",
    "Area Code (M49)",
    "M49 Code",
    "Area",
    "Item Code",
    "Item Code (FBS)",
    "Item",
    "Year",
]

population_columns = [
    column
    for column in [
        "Population 1000",
        "Population unit",
        "Population flag",
        "Population",
    ]
    if column in supervised_panel.columns
]

target_columns = [
    "target_year",
    "Temporal split",
    "shortage_next_year",
]

# A value column has a corresponding quality-flag column
flag_columns = sorted(
    column
    for column in supervised_panel.columns
    if column.endswith("_flag")
    and column[:-5] in supervised_panel.columns
)

value_columns = [
    column[:-5]
    for column in flag_columns
]

source_presence_columns = [
    f"{column}_source_present"
    for column in value_columns
    if f"{column}_source_present" in supervised_panel.columns
]

already_classified = set(
    identity_columns
    + population_columns
    + target_columns
    + value_columns
    + flag_columns
    + source_presence_columns
)

quality_profile_columns = [
    column
    for column in supervised_panel.columns
    if column not in already_classified
]

column_structure = pd.DataFrame(
    {
        "Column group": [
            "Identity and time",
            "Analytical values",
            "Quality flags",
            "Source-presence indicators",
            "Population information",
            "Row-quality and descriptive fields",
            "Target and temporal split",
        ],
        "Number of columns": [
            len(identity_columns),
            len(value_columns),
            len(flag_columns),
            len(source_presence_columns),
            len(population_columns),
            len(quality_profile_columns),
            len(target_columns),
        ],
    }
)

print("Supervised-panel column structure:")
display(column_structure)

print("\nAnalytical value columns:")
for number, column in enumerate(value_columns, start=1):
    print(f"{number:>2}. {column}")

print("\nRow-quality and descriptive columns:")
for number, column in enumerate(quality_profile_columns, start=1):
    print(f"{number:>2}. {column}")

# ---------------------------------------------------------
# Validate the saved target
# ---------------------------------------------------------

target_column = target_definition.loc[0, "Target column"]

target_audit = pd.Series(
    {
        "Target column": target_column,
        "Rows": len(supervised_panel),
        "Missing target values": (
            supervised_panel[target_column].isna().sum()
        ),
        "Unique target values": (
            supervised_panel[target_column].nunique()
        ),
        "Positive events": (
            supervised_panel[target_column].eq(1).sum()
        ),
        "Negative outcomes": (
            supervised_panel[target_column].eq(0).sum()
        ),
        "Event rate %": (
            supervised_panel[target_column].mean() * 100
        ),
        "Target year equals predictor year plus one": (
            supervised_panel["target_year"]
            .eq(supervised_panel["Year"] + 1)
            .all()
        ),
        "Duplicate country-commodity-year keys": (
            supervised_panel[
                ["Area", "Item Code", "Year"]
            ].duplicated().sum()
        ),
    },
    name="Result",
).to_frame()

print("\nTarget audit:")
display(target_audit)

split_target_audit = (
    supervised_panel
    .groupby("Temporal split", observed=True)
    .agg(
        Predictor_year_start=("Year", "min"),
        Predictor_year_end=("Year", "max"),
        Target_year_start=("target_year", "min"),
        Target_year_end=("target_year", "max"),
        Eligible_rows=(target_column, "size"),
        Positive_events=(target_column, "sum"),
        Countries=("Area", "nunique"),
        Commodities=("Item", "nunique"),
    )
    .reset_index()
)

split_target_audit["Negative_events"] = (
    split_target_audit["Eligible_rows"]
    - split_target_audit["Positive_events"]
)

split_target_audit["Event rate %"] = (
    100
    * split_target_audit["Positive_events"]
    / split_target_audit["Eligible_rows"]
)

split_order = ["Train", "Validation", "Test"]

split_target_audit["Temporal split"] = pd.Categorical(
    split_target_audit["Temporal split"],
    categories=split_order,
    ordered=True,
)

split_target_audit = (
    split_target_audit
    .sort_values("Temporal split")
    .reset_index(drop=True)
)

print("\nReproduced target balance by temporal split:")
display(split_target_audit)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert supervised_panel.shape == (3248, 87)
assert target_column == "shortage_next_year"
assert supervised_panel[target_column].notna().all()
assert set(supervised_panel[target_column].unique()) == {0, 1}
assert supervised_panel["target_year"].eq(
    supervised_panel["Year"] + 1
).all()

assert (
    supervised_panel[
        ["Area", "Item Code", "Year"]
    ].duplicated().sum()
    == 0
)

assert len(value_columns) == 20
assert len(flag_columns) == 20
assert len(source_presence_columns) == 20

assert split_target_audit["Eligible_rows"].tolist() == [
    2250,
    500,
    498,
]

assert split_target_audit["Positive_events"].tolist() == [
    231,
    52,
    51,
]

print(
    "\nThe supervised-panel structure and "
    "next-year shortage target were validated successfully."
)

Supervised-panel column structure:


,Column group,Number of columns
0,Identity and time,8
1,Analytical values,20
2,Quality flags,20
3,Source-presence indicators,20
4,Population information,4
5,Row-quality and descriptive fields,12
6,Target and temporal split,3



Analytical value columns:
 1. domestic_supply_1000t
 2. export_quantity_1000t
 3. fat_supply_g_cap_day
 4. fat_supply_t
 5. feed_1000t
 6. food_1000t
 7. food_supply_kcal_cap_day
 8. food_supply_million_kcal
 9. food_supply_quantity_kg_cap_yr
10. import_quantity_1000t
11. losses_1000t
12. other_uses_non_food_1000t
13. processing_1000t
14. production_1000t
15. protein_supply_g_cap_day
16. protein_supply_t
17. residuals_1000t
18. seed_1000t
19. stock_variation_1000t
20. tourist_consumption_1000t

Row-quality and descriptive columns:
 1. recorded_element_count
 2. source_present_element_count
 3. source_absent_element_count
 4. source_present_missing_element_count
 5. estimated_element_count
 6. imputed_element_count
 7. recorded_zero_element_count
 8. recorded_negative_element_count
 9. supply_outcome_recorded_count
10. supply_outcomes_complete
11. production_status
12. negative_domestic_supply_flag

Target audit:


,Result
Target column,shortage_next_year
Rows,3248
Missing target values,0
Unique target values,2
Positive events,334
Negative outcomes,2914
Event rate %,10.28
Target year equals predictor year plus one,True
Duplicate country-commodity-year keys,0



Reproduced target balance by temporal split:


,Temporal split,Predictor_year_start,Predictor_year_end,Target_year_start,Target_year_end,Eligible_rows,Positive_events,Countries,Commodities,Negative_events,Event rate %
0,Train,2010,2018,2011,2019,2250,231,43,8,2019,10.27
1,Validation,2019,2020,2020,2021,500,52,43,8,448,10.40
2,Test,2021,2022,2022,2023,498,51,43,8,447,10.24



The supervised-panel structure and next-year shortage target were validated successfully.


In [4]:
# ---------------------------------------------------------
# Define the modelling keys and target information
# ---------------------------------------------------------

panel_key_columns = [
    "Area",
    "Item Code",
    "Year",
]

target_column = "shortage_next_year"

target_information_columns = [
    "target_year",
    "Temporal split",
    target_column,
]

# The modelling index preserves the observations that are
# eligible for supervised learning.
model_index = supervised_panel[
    [
        "Area Code",
        "Area Code (M49)",
        "M49 Code",
        "Area",
        "Item Code",
        "Item Code (FBS)",
        "Item",
        "Year",
        "target_year",
        "Temporal split",
        target_column,
    ]
].copy()

# Historical features must be calculated from the complete
# analytical panel, not from the filtered supervised panel.
feature_source = (
    analytical_panel
    .sort_values(
        ["Area", "Item Code", "Year"]
    )
    .reset_index(drop=True)
    .copy()
)

# ---------------------------------------------------------
# Audit the country-commodity timelines
# ---------------------------------------------------------

pair_timeline_audit = (
    feature_source
    .groupby(
        ["Area", "Item Code", "Item"],
        observed=True,
    )
    .agg(
        Rows=("Year", "size"),
        Unique_years=("Year", "nunique"),
        First_year=("Year", "min"),
        Last_year=("Year", "max"),
    )
    .reset_index()
)

# Calculate the gap between each row and its preceding year
feature_source["_previous_year_for_audit"] = (
    feature_source
    .groupby(
        ["Area", "Item Code"],
        observed=True,
    )["Year"]
    .shift(1)
)

feature_source["_year_gap_for_audit"] = (
    feature_source["Year"]
    - feature_source["_previous_year_for_audit"]
)

unexpected_year_gaps = feature_source[
    feature_source["_previous_year_for_audit"].notna()
    & feature_source["_year_gap_for_audit"].ne(1)
].copy()

feature_source = feature_source.drop(
    columns=[
        "_previous_year_for_audit",
        "_year_gap_for_audit",
    ]
)

timeline_summary = pd.Series(
    {
        "Complete-panel rows": len(feature_source),
        "Countries": feature_source["Area"].nunique(),
        "Commodities": feature_source["Item"].nunique(),
        "Country-commodity pairs": len(pair_timeline_audit),
        "Minimum rows per pair": pair_timeline_audit["Rows"].min(),
        "Maximum rows per pair": pair_timeline_audit["Rows"].max(),
        "Minimum years per pair": (
            pair_timeline_audit["Unique_years"].min()
        ),
        "Maximum years per pair": (
            pair_timeline_audit["Unique_years"].max()
        ),
        "Earliest year": feature_source["Year"].min(),
        "Latest year": feature_source["Year"].max(),
        "Unexpected gaps between years": (
            len(unexpected_year_gaps)
        ),
        "Duplicate country-commodity-year keys": (
            feature_source[
                panel_key_columns
            ].duplicated().sum()
        ),
    },
    name="Result",
).to_frame()

print("Complete feature-source timeline audit:")
display(timeline_summary)

# ---------------------------------------------------------
# Confirm that every supervised row exists in the full panel
# ---------------------------------------------------------

supervised_key_check = model_index[
    panel_key_columns
].merge(
    feature_source[panel_key_columns],
    on=panel_key_columns,
    how="left",
    validate="one_to_one",
    indicator=True,
)

missing_supervised_keys = supervised_key_check[
    supervised_key_check["_merge"].ne("both")
]

key_match_summary = pd.Series(
    {
        "Supervised rows requiring features": len(model_index),
        "Supervised keys found in complete panel": (
            supervised_key_check["_merge"].eq("both").sum()
        ),
        "Supervised keys missing from complete panel": (
            len(missing_supervised_keys)
        ),
    },
    name="Result",
).to_frame()

print("\nSupervised-to-complete-panel key check:")
display(key_match_summary)

# ---------------------------------------------------------
# Record the feature-use policy
# ---------------------------------------------------------

feature_policy = pd.DataFrame(
    {
        "Information group": [
            "Current-year analytical values",
            "Earlier-year analytical values",
            "Current and earlier quality information",
            "Country and commodity identities",
            "Predictor year",
            "Next-year shortage target",
            "Target year",
            "Temporal split label",
            "Next-year analytical values",
        ],
        "Use in feature creation": [
            "Yes",
            "Yes",
            "Yes",
            "Yes, after suitable encoding",
            "Yes",
            "No",
            "No",
            "No",
            "No",
        ],
        "Reason": [
            "Known at the point from which the forecast is made",
            "Provides historical direction and recent behaviour",
            "Describes the reliability and completeness of known data",
            "Allows the model to recognise geographical and commodity differences",
            "Allows long-term time movement to be represented",
            "This is the answer the model must predict",
            "It identifies the future outcome year",
            "It is used for evaluation control, not prediction",
            "These values would not yet be known when making the forecast",
        ],
    }
)

print("\nLeakage-safe feature policy:")
display(feature_policy)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert feature_source.shape[0] == 4816
assert len(pair_timeline_audit) == 344

assert pair_timeline_audit["Rows"].eq(14).all()
assert pair_timeline_audit["Unique_years"].eq(14).all()
assert pair_timeline_audit["First_year"].eq(2010).all()
assert pair_timeline_audit["Last_year"].eq(2023).all()

assert len(unexpected_year_gaps) == 0

assert (
    feature_source[
        panel_key_columns
    ].duplicated().sum()
    == 0
)

assert len(missing_supervised_keys) == 0

assert target_column not in feature_source.columns
assert "target_year" not in feature_source.columns
assert "Temporal split" not in feature_source.columns

print(
    "\nThe complete analytical panel is a valid, "
    "continuous and leakage-safe source for feature creation."
)

Complete feature-source timeline audit:


,Result
Complete-panel rows,4816
Countries,43
Commodities,8
Country-commodity pairs,344
Minimum rows per pair,14
Maximum rows per pair,14
Minimum years per pair,14
Maximum years per pair,14
Earliest year,2010
Latest year,2023



Supervised-to-complete-panel key check:


,Result
Supervised rows requiring features,3248
Supervised keys found in complete panel,3248
Supervised keys missing from complete panel,0



Leakage-safe feature policy:


,Information group,Use in feature creation,Reason
0,Current-year analytical values,Yes,Known at the point from which the forecast is ...
1,Earlier-year analytical values,Yes,Provides historical direction and recent behav...
2,Current and earlier quality information,Yes,Describes the reliability and completeness of ...
3,Country and commodity identities,"Yes, after suitable encoding",Allows the model to recognise geographical and...
4,Predictor year,Yes,Allows long-term time movement to be represented
5,Next-year shortage target,No,This is the answer the model must predict
6,Target year,No,It identifies the future outcome year
7,Temporal split label,No,"It is used for evaluation control, not prediction"
8,Next-year analytical values,No,These values would not yet be known when makin...



The complete analytical panel is a valid, continuous and leakage-safe source for feature creation.


In [5]:
# ---------------------------------------------------------
# Define the initial current-year analytical predictors
# ---------------------------------------------------------

derived_total_columns = [
    "food_supply_million_kcal",
    "protein_supply_t",
    "fat_supply_t",
]

current_analytical_features = [
    column
    for column in value_columns
    if column not in derived_total_columns
]

split_order = ["Train", "Validation", "Test"]

# ---------------------------------------------------------
# Audit analytical-value availability and behaviour
# ---------------------------------------------------------

analytical_inventory_rows = []

for column in value_columns:
    recorded_mask = supervised_panel[column].notna()
    recorded_values = supervised_panel.loc[
        recorded_mask,
        column,
    ]

    split_coverage = {}

    for split_name in split_order:
        split_mask = (
            supervised_panel["Temporal split"]
            .eq(split_name)
        )

        split_coverage[split_name] = (
            100
            * supervised_panel.loc[
                split_mask,
                column,
            ].notna().mean()
        )

    matching_flag_column = f"{column}_flag"

    unique_flags = (
        supervised_panel.loc[
            supervised_panel[
                matching_flag_column
            ].notna(),
            matching_flag_column,
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    unique_flags = ", ".join(
        sorted(unique_flags)
    )

    if column in derived_total_columns:
        modelling_decision = (
            "Excluded from primary predictor set"
        )
        decision_reason = (
            "National total duplicates information "
            "already represented by a per-person "
            "measure and population"
        )
    else:
        modelling_decision = (
            "Retained as a current-year predictor"
        )
        decision_reason = (
            "Represents a distinct nutritional, "
            "supply, trade or utilisation measurement"
        )

    analytical_inventory_rows.append(
        {
            "Analytical column": column,
            "Recorded values": recorded_mask.sum(),
            "Overall coverage %": (
                100 * recorded_mask.mean()
            ),
            "Train coverage %": (
                split_coverage["Train"]
            ),
            "Validation coverage %": (
                split_coverage["Validation"]
            ),
            "Test coverage %": (
                split_coverage["Test"]
            ),
            "Zero % of recorded": (
                100
                * recorded_values.eq(0).mean()
                if len(recorded_values)
                else np.nan
            ),
            "Negative % of recorded": (
                100
                * recorded_values.lt(0).mean()
                if len(recorded_values)
                else np.nan
            ),
            "Observed flags": unique_flags,
            "Modelling decision": modelling_decision,
            "Reason": decision_reason,
        }
    )

analytical_predictor_inventory = pd.DataFrame(
    analytical_inventory_rows
)

print("Current-year analytical predictor inventory:")
display(analytical_predictor_inventory)

# ---------------------------------------------------------
# Audit the row-quality fields
# ---------------------------------------------------------

quality_inventory_rows = []

for column in quality_profile_columns:
    non_missing_values = (
        supervised_panel[column]
        .dropna()
    )

    example_values = (
        non_missing_values
        .astype(str)
        .drop_duplicates()
        .head(5)
        .tolist()
    )

    quality_inventory_rows.append(
        {
            "Quality column": column,
            "Data type": str(
                supervised_panel[column].dtype
            ),
            "Missing values": (
                supervised_panel[column]
                .isna()
                .sum()
            ),
            "Unique non-missing values": (
                non_missing_values.nunique()
            ),
            "Example values": ", ".join(
                example_values
            ),
        }
    )

quality_predictor_inventory = pd.DataFrame(
    quality_inventory_rows
)

print("\nRow-quality predictor inventory:")
display(quality_predictor_inventory)

# ---------------------------------------------------------
# Summarise the initial feature decision
# ---------------------------------------------------------

initial_feature_decision = pd.Series(
    {
        "Available analytical measurements": (
            len(value_columns)
        ),
        "Current analytical predictors retained": (
            len(current_analytical_features)
        ),
        "Derived national totals excluded": (
            len(derived_total_columns)
        ),
        "Population measure to retain": (
            "Population"
        ),
        "Country identity retained": (
            "Area"
        ),
        "Commodity identity retained": (
            "Item Code"
        ),
        "Predictor year retained": (
            "Year"
        ),
    },
    name="Decision",
).to_frame()

print("\nInitial current-year feature decision:")
display(initial_feature_decision)

print("\nRetained analytical predictors:")

for number, column in enumerate(
    current_analytical_features,
    start=1,
):
    print(f"{number:>2}. {column}")

print("\nExcluded derived national totals:")

for number, column in enumerate(
    derived_total_columns,
    start=1,
):
    print(f"{number:>2}. {column}")

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(value_columns) == 20
assert len(current_analytical_features) == 17
assert len(derived_total_columns) == 3

assert set(current_analytical_features).isdisjoint(
    derived_total_columns
)

assert set(
    current_analytical_features
    + derived_total_columns
) == set(value_columns)

assert supervised_panel[
    current_analytical_features
].shape[1] == 17

print(
    "\nThe current-year analytical predictor "
    "inventory was completed successfully."
)

Current-year analytical predictor inventory:


,Analytical column,Recorded values,Overall coverage %,Train coverage %,Validation coverage %,Test coverage %,Zero % of recorded,Negative % of recorded,Observed flags,Modelling decision,Reason
0,domestic_supply_1000t,3248,100.00,100.00,100.00,100.00,0.55,0.00,I,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."
1,export_quantity_1000t,2555,78.66,78.84,77.20,79.32,48.22,0.00,I,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."
2,fat_supply_g_cap_day,3248,100.00,100.00,100.00,100.00,0.00,0.00,E,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."
3,fat_supply_t,3248,100.00,100.00,100.00,100.00,0.00,0.00,I,Excluded from primary predictor set,National total duplicates information already ...
4,feed_1000t,2646,81.47,80.53,83.00,84.14,30.57,0.00,I,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."
5,food_1000t,3248,100.00,100.00,100.00,100.00,0.68,0.00,I,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."
6,food_supply_kcal_cap_day,3248,100.00,100.00,100.00,100.00,0.00,0.00,E,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."
7,food_supply_million_kcal,3248,100.00,100.00,100.00,100.00,0.00,0.00,I,Excluded from primary predictor set,National total duplicates information already ...
8,food_supply_quantity_kg_cap_yr,3248,100.00,100.00,100.00,100.00,0.00,0.00,E,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."
9,import_quantity_1000t,3066,94.40,93.82,95.20,96.18,24.10,0.00,I,Retained as a current-year predictor,"Represents a distinct nutritional, supply, tra..."



Row-quality predictor inventory:


,Quality column,Data type,Missing values,Unique non-missing values,Example values
0,recorded_element_count,int8,0,10,"18, 19, 17, 16, 15"
1,source_present_element_count,int8,0,9,"19, 18, 17, 16, 15"
2,source_absent_element_count,int8,0,9,"1, 2, 3, 4, 5"
3,source_present_missing_element_count,int8,0,5,"1, 0, 2, 3, 4"
4,estimated_element_count,int8,0,1,4
5,imputed_element_count,int8,0,10,"14, 15, 13, 12, 11"
6,recorded_zero_element_count,int8,0,10,"1, 0, 2, 4, 3"
7,recorded_negative_element_count,int8,0,3,"1, 0, 2"
8,supply_outcome_recorded_count,int8,0,2,"5, 4"
9,supply_outcomes_complete,bool,0,2,"True, False"



Initial current-year feature decision:


,Decision
Available analytical measurements,20
Current analytical predictors retained,17
Derived national totals excluded,3
Population measure to retain,Population
Country identity retained,Area
Commodity identity retained,Item Code
Predictor year retained,Year



Retained analytical predictors:
 1. domestic_supply_1000t
 2. export_quantity_1000t
 3. fat_supply_g_cap_day
 4. feed_1000t
 5. food_1000t
 6. food_supply_kcal_cap_day
 7. food_supply_quantity_kg_cap_yr
 8. import_quantity_1000t
 9. losses_1000t
10. other_uses_non_food_1000t
11. processing_1000t
12. production_1000t
13. protein_supply_g_cap_day
14. residuals_1000t
15. seed_1000t
16. stock_variation_1000t
17. tourist_consumption_1000t

Excluded derived national totals:
 1. food_supply_million_kcal
 2. protein_supply_t
 3. fat_supply_t

The current-year analytical predictor inventory was completed successfully.


In [6]:
# ---------------------------------------------------------
# Use training data only to decide which measurements
# have sufficient coverage for historical features
# ---------------------------------------------------------

training_mask = (
    supervised_panel["Temporal split"]
    .eq("Train")
)

historical_coverage_threshold = 70.0

training_coverage = (
    supervised_panel.loc[
        training_mask,
        current_analytical_features,
    ]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

historical_base_features = [
    column
    for column in current_analytical_features
    if training_coverage[column]
    >= historical_coverage_threshold
]

sparse_current_only_features = [
    column
    for column in current_analytical_features
    if column not in historical_base_features
]

historical_feature_decision = pd.DataFrame(
    {
        "Analytical feature": (
            current_analytical_features
        ),
        "Training coverage %": [
            training_coverage[column]
            for column in current_analytical_features
        ],
        "Retained as current-year feature": True,
        "Historical features created": [
            column in historical_base_features
            for column in current_analytical_features
        ],
    }
)

historical_feature_decision[
    "Decision explanation"
] = np.where(
    historical_feature_decision[
        "Historical features created"
    ],
    (
        "Training coverage is at least 70%; "
        "previous-year and recent-history "
        "features will be created"
    ),
    (
        "Retained in current form, but historical "
        "features are avoided because training "
        "coverage is below 70%"
    ),
)

print("Historical-feature coverage decision:")
display(historical_feature_decision)

# ---------------------------------------------------------
# Select non-redundant row-quality predictors
# ---------------------------------------------------------

numeric_quality_features = [
    "recorded_element_count",
    "source_absent_element_count",
    "source_present_missing_element_count",
    "recorded_zero_element_count",
    "recorded_negative_element_count",
    "supply_outcome_recorded_count",
]

categorical_quality_features = [
    "production_status",
]

excluded_quality_features = {
    "source_present_element_count": (
        "It equals 20 minus the source-absent count"
    ),
    "estimated_element_count": (
        "It is constant at four in the supervised panel"
    ),
    "imputed_element_count": (
        "It is determined by recorded values minus "
        "estimated values"
    ),
    "supply_outcomes_complete": (
        "It repeats whether the supply-outcome "
        "recorded count equals five"
    ),
    "negative_domestic_supply_flag": (
        "It is always False in the supervised panel"
    ),
}

quality_feature_decision_rows = []

for column in quality_profile_columns:
    if column in numeric_quality_features:
        decision = "Retained as numeric quality feature"
        reason = (
            "Provides distinct information about "
            "data availability or recorded-value behaviour"
        )

    elif column in categorical_quality_features:
        decision = "Retained as categorical quality feature"
        reason = (
            "Distinguishes positive, zero, missing and "
            "structurally absent production"
        )

    else:
        decision = "Excluded as redundant or constant"
        reason = excluded_quality_features[column]

    quality_feature_decision_rows.append(
        {
            "Quality column": column,
            "Decision": decision,
            "Reason": reason,
        }
    )

quality_feature_decision = pd.DataFrame(
    quality_feature_decision_rows
)

print("\nQuality-feature decision:")
display(quality_feature_decision)

# ---------------------------------------------------------
# Define the overall feature blueprint
# ---------------------------------------------------------

identity_features = [
    "Area",
    "Item Code",
]

time_features = [
    "Year",
]

population_features = [
    "Population",
]

# These names will be created in later blocks.
missing_indicator_features = [
    f"{column}_missing"
    for column in current_analytical_features
]

source_indicator_features = [
    f"{column}_source_present"
    for column in current_analytical_features
]

historical_feature_types = [
    "Previous-year value",
    "One-year absolute change",
    "One-year percentage change",
    "Three-year average",
    "Three-year variability",
]

feature_blueprint_summary = pd.Series(
    {
        "Identity features": len(identity_features),
        "Time features": len(time_features),
        "Population features": len(population_features),
        "Current analytical features": (
            len(current_analytical_features)
        ),
        "Analytical features receiving history": (
            len(historical_base_features)
        ),
        "Sparse current-only analytical features": (
            len(sparse_current_only_features)
        ),
        "Missing-value indicators to create": (
            len(missing_indicator_features)
        ),
        "Source-presence indicators to retain": (
            len(source_indicator_features)
        ),
        "Numeric row-quality features": (
            len(numeric_quality_features)
        ),
        "Categorical row-quality features": (
            len(categorical_quality_features)
        ),
    },
    name="Result",
).to_frame()

print("\nFeature blueprint summary:")
display(feature_blueprint_summary)

print("\nMeasurements receiving historical features:")

for number, column in enumerate(
    historical_base_features,
    start=1,
):
    print(f"{number:>2}. {column}")

print("\nSparse measurements retained only in current form:")

for number, column in enumerate(
    sparse_current_only_features,
    start=1,
):
    print(f"{number:>2}. {column}")

print("\nHistorical feature types that will be created:")

for number, description in enumerate(
    historical_feature_types,
    start=1,
):
    print(f"{number}. {description}")

# ---------------------------------------------------------
# Confirm the reasons for excluding redundant fields
# ---------------------------------------------------------

assert (
    supervised_panel[
        "source_present_element_count"
    ]
    + supervised_panel[
        "source_absent_element_count"
    ]
).eq(20).all()

assert (
    supervised_panel[
        "source_present_element_count"
    ]
    - supervised_panel[
        "recorded_element_count"
    ]
).eq(
    supervised_panel[
        "source_present_missing_element_count"
    ]
).all()

assert (
    supervised_panel[
        "estimated_element_count"
    ]
    + supervised_panel[
        "imputed_element_count"
    ]
).eq(
    supervised_panel[
        "recorded_element_count"
    ]
).all()

assert supervised_panel[
    "estimated_element_count"
].eq(4).all()

assert supervised_panel[
    "supply_outcomes_complete"
].eq(
    supervised_panel[
        "supply_outcome_recorded_count"
    ].eq(5)
).all()

assert (
    supervised_panel[
        "negative_domestic_supply_flag"
    ].nunique()
    == 1
)

# Every analytical flag has only one non-missing category
flag_unique_counts = (
    supervised_panel[flag_columns]
    .nunique(dropna=True)
)

assert flag_unique_counts.eq(1).all()

assert len(historical_base_features) == 14

assert set(sparse_current_only_features) == {
    "other_uses_non_food_1000t",
    "processing_1000t",
    "tourist_consumption_1000t",
}

assert target_column not in (
    identity_features
    + time_features
    + population_features
    + current_analytical_features
    + numeric_quality_features
    + categorical_quality_features
)

print(
    "\nThe modelling feature blueprint was "
    "defined and validated successfully."
)

Historical-feature coverage decision:


,Analytical feature,Training coverage %,Retained as current-year feature,Historical features created,Decision explanation
0,domestic_supply_1000t,100.00,True,True,Training coverage is at least 70%; previous-ye...
1,export_quantity_1000t,78.84,True,True,Training coverage is at least 70%; previous-ye...
2,fat_supply_g_cap_day,100.00,True,True,Training coverage is at least 70%; previous-ye...
3,feed_1000t,80.53,True,True,Training coverage is at least 70%; previous-ye...
4,food_1000t,100.00,True,True,Training coverage is at least 70%; previous-ye...
5,food_supply_kcal_cap_day,100.00,True,True,Training coverage is at least 70%; previous-ye...
6,food_supply_quantity_kg_cap_yr,100.00,True,True,Training coverage is at least 70%; previous-ye...
7,import_quantity_1000t,93.82,True,True,Training coverage is at least 70%; previous-ye...
8,losses_1000t,90.31,True,True,Training coverage is at least 70%; previous-ye...
9,other_uses_non_food_1000t,36.53,True,False,"Retained in current form, but historical featu..."



Quality-feature decision:


,Quality column,Decision,Reason
0,recorded_element_count,Retained as numeric quality feature,Provides distinct information about data avail...
1,source_present_element_count,Excluded as redundant or constant,It equals 20 minus the source-absent count
2,source_absent_element_count,Retained as numeric quality feature,Provides distinct information about data avail...
3,source_present_missing_element_count,Retained as numeric quality feature,Provides distinct information about data avail...
4,estimated_element_count,Excluded as redundant or constant,It is constant at four in the supervised panel
5,imputed_element_count,Excluded as redundant or constant,It is determined by recorded values minus esti...
6,recorded_zero_element_count,Retained as numeric quality feature,Provides distinct information about data avail...
7,recorded_negative_element_count,Retained as numeric quality feature,Provides distinct information about data avail...
8,supply_outcome_recorded_count,Retained as numeric quality feature,Provides distinct information about data avail...
9,supply_outcomes_complete,Excluded as redundant or constant,It repeats whether the supply-outcome recorded...



Feature blueprint summary:


,Result
Identity features,2
Time features,1
Population features,1
Current analytical features,17
Analytical features receiving history,14
Sparse current-only analytical features,3
Missing-value indicators to create,17
Source-presence indicators to retain,17
Numeric row-quality features,6
Categorical row-quality features,1



Measurements receiving historical features:
 1. domestic_supply_1000t
 2. export_quantity_1000t
 3. fat_supply_g_cap_day
 4. feed_1000t
 5. food_1000t
 6. food_supply_kcal_cap_day
 7. food_supply_quantity_kg_cap_yr
 8. import_quantity_1000t
 9. losses_1000t
10. production_1000t
11. protein_supply_g_cap_day
12. residuals_1000t
13. seed_1000t
14. stock_variation_1000t

Sparse measurements retained only in current form:
 1. other_uses_non_food_1000t
 2. processing_1000t
 3. tourist_consumption_1000t

Historical feature types that will be created:
1. Previous-year value
2. One-year absolute change
3. One-year percentage change
4. Three-year average
5. Three-year variability

The modelling feature blueprint was defined and validated successfully.


In [7]:
# ---------------------------------------------------------
# Confirm that all required source indicators exist
# ---------------------------------------------------------

missing_source_indicator_columns = [
    column
    for column in source_indicator_features
    if column not in feature_source.columns
]

if missing_source_indicator_columns:
    raise KeyError(
        "The following required source-presence "
        "columns are missing:\n"
        + "\n".join(
            f"- {column}"
            for column in missing_source_indicator_columns
        )
    )

# ---------------------------------------------------------
# Build a feature-engineering panel on the complete grid
# ---------------------------------------------------------

feature_engineering_columns = (
    panel_key_columns
    + population_features
    + current_analytical_features
    + numeric_quality_features
    + categorical_quality_features
    + source_indicator_features
)

feature_engineering_panel = (
    feature_source[
        feature_engineering_columns
    ]
    .copy()
    .sort_values(
        ["Area", "Item Code", "Year"]
    )
    .reset_index(drop=True)
)

# Create one missing-value indicator for each retained
# current analytical measurement.
for column in current_analytical_features:
    missing_column = f"{column}_missing"

    feature_engineering_panel[
        missing_column
    ] = (
        feature_engineering_panel[column]
        .isna()
        .astype("int8")
    )

# ---------------------------------------------------------
# Attach features only to target-eligible modelling rows
# ---------------------------------------------------------

feature_payload_columns = (
    panel_key_columns
    + population_features
    + current_analytical_features
    + numeric_quality_features
    + categorical_quality_features
    + source_indicator_features
    + missing_indicator_features
)

model_current_features = (
    model_index
    .merge(
        feature_engineering_panel[
            feature_payload_columns
        ],
        on=panel_key_columns,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["Area", "Item Code", "Year"]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Summarise current analytical-value availability
# ---------------------------------------------------------

current_value_status_rows = []

for split_name in split_order:
    split_mask = (
        model_current_features[
            "Temporal split"
        ].eq(split_name)
    )

    split_values = model_current_features.loc[
        split_mask,
        current_analytical_features,
    ]

    split_sources = model_current_features.loc[
        split_mask,
        source_indicator_features,
    ].astype(bool)

    value_missing_matrix = (
        split_values.isna()
    )

    total_cells = split_values.size

    recorded_cells = (
        split_values.notna()
        .to_numpy()
        .sum()
    )

    source_absent_cells = (
        (~split_sources)
        .to_numpy()
        .sum()
    )

    source_present_missing_cells = (
        value_missing_matrix.to_numpy()
        & split_sources.to_numpy()
    ).sum()

    current_value_status_rows.append(
        {
            "Temporal split": split_name,
            "Rows": split_mask.sum(),
            "Possible current analytical cells": (
                total_cells
            ),
            "Recorded cells": recorded_cells,
            "Source-present missing cells": (
                source_present_missing_cells
            ),
            "Source-absent cells": (
                source_absent_cells
            ),
            "Recorded coverage %": (
                100
                * recorded_cells
                / total_cells
            ),
        }
    )

current_value_status_summary = pd.DataFrame(
    current_value_status_rows
)

current_value_status_summary[
    "Temporal split"
] = pd.Categorical(
    current_value_status_summary[
        "Temporal split"
    ],
    categories=split_order,
    ordered=True,
)

current_value_status_summary = (
    current_value_status_summary
    .sort_values("Temporal split")
    .reset_index(drop=True)
)

print("Current-year analytical-value status:")
display(current_value_status_summary)

# ---------------------------------------------------------
# Summarise the constructed table
# ---------------------------------------------------------

current_feature_table_summary = pd.Series(
    {
        "Rows": len(model_current_features),
        "Columns": model_current_features.shape[1],
        "Countries": (
            model_current_features["Area"].nunique()
        ),
        "Commodities": (
            model_current_features["Item"].nunique()
        ),
        "Predictor years": (
            model_current_features["Year"].nunique()
        ),
        "Current analytical features": (
            len(current_analytical_features)
        ),
        "Missing-value indicators": (
            len(missing_indicator_features)
        ),
        "Source-presence indicators": (
            len(source_indicator_features)
        ),
        "Numeric quality features": (
            len(numeric_quality_features)
        ),
        "Categorical quality features": (
            len(categorical_quality_features)
        ),
        "Missing target values": (
            model_current_features[
                target_column
            ].isna().sum()
        ),
        "Duplicate modelling keys": (
            model_current_features[
                panel_key_columns
            ].duplicated().sum()
        ),
    },
    name="Result",
).to_frame()

print("\nCurrent-year modelling table summary:")
display(current_feature_table_summary)

print("\nFirst five modelling rows — selected fields:")

display(
    model_current_features[
        [
            "Area",
            "Item",
            "Year",
            "Population",
            "food_supply_kcal_cap_day",
            "food_supply_kcal_cap_day_missing",
            "production_1000t",
            "production_1000t_missing",
            "production_status",
            "Temporal split",
            target_column,
        ]
    ].head()
)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(feature_engineering_panel) == 4816
assert len(model_current_features) == 3248
assert model_current_features.shape[1] == 70

assert (
    model_current_features[
        panel_key_columns
    ].duplicated().sum()
    == 0
)

assert model_current_features[
    target_column
].notna().all()

assert (
    model_current_features[
        target_column
    ].sum()
    == 334
)

# Confirm every missing indicator matches its value column.
for column in current_analytical_features:
    missing_column = f"{column}_missing"

    assert model_current_features[
        missing_column
    ].eq(
        model_current_features[
            column
        ].isna()
        .astype("int8")
    ).all()

    source_column = (
        f"{column}_source_present"
    )

    # A recorded value must have a source row.
    assert model_current_features.loc[
        model_current_features[
            column
        ].notna(),
        source_column,
    ].astype(bool).all()

    # A source-absent cell cannot contain a value.
    assert model_current_features.loc[
        ~model_current_features[
            source_column
        ].astype(bool),
        column,
    ].isna().all()

# Confirm the three cell statuses account for every cell.
assert (
    current_value_status_summary[
        "Recorded cells"
    ]
    + current_value_status_summary[
        "Source-present missing cells"
    ]
    + current_value_status_summary[
        "Source-absent cells"
    ]
).eq(
    current_value_status_summary[
        "Possible current analytical cells"
    ]
).all()

print(
    "\nThe current-year modelling feature table "
    "was created and validated successfully."
)

Current-year analytical-value status:


,Temporal split,Rows,Possible current analytical cells,Recorded cells,Source-present missing cells,Source-absent cells,Recorded coverage %
0,Train,2250,38250,31594,1354,5302,82.60
1,Validation,500,8500,6967,350,1183,81.96
2,Test,498,8466,6970,326,1170,82.33



Current-year modelling table summary:


,Result
Rows,3248
Columns,70
Countries,43
Commodities,8
Predictor years,13
Current analytical features,17
Missing-value indicators,17
Source-presence indicators,17
Numeric quality features,6
Categorical quality features,1



First five modelling rows — selected fields:


,Area,Item,Year,Population,food_supply_kcal_cap_day,food_supply_kcal_cap_day_missing,production_1000t,production_1000t_missing,production_status,Temporal split,shortage_next_year
0,Algeria,Wheat and products,2010,"36,188,240.00","1,314.96",0,"2,605.00",0,Recorded positive production,Train,0
1,Algeria,Wheat and products,2011,"36,903,380.00","1,287.02",0,"2,911.00",0,Recorded positive production,Train,0
2,Algeria,Wheat and products,2012,"37,646,170.00","1,312.79",0,"3,432.00",0,Recorded positive production,Train,0
3,Algeria,Wheat and products,2013,"38,414,170.00","1,270.79",0,"3,299.00",0,Recorded positive production,Train,0
4,Algeria,Wheat and products,2014,"39,205,030.00","1,252.06",0,"2,436.00",0,Recorded positive production,Train,0



The current-year modelling feature table was created and validated successfully.


In [8]:
# ---------------------------------------------------------
# Ensure the complete feature panel is in chronological order
# ---------------------------------------------------------

feature_engineering_panel = (
    feature_engineering_panel
    .sort_values(
        ["Area", "Item Code", "Year"]
    )
    .reset_index(drop=True)
)

history_group_columns = [
    "Area",
    "Item Code",
]

historical_feature_columns = []

lag1_columns = []
change1_columns = []
percentage_change1_columns = []
rolling3_mean_columns = []
rolling3_std_columns = []

# ---------------------------------------------------------
# Create historical features for the 14 sufficiently
# available analytical measurements
# ---------------------------------------------------------

for column in historical_base_features:
    lag_column = f"{column}_lag1"
    change_column = f"{column}_change1"
    percentage_column = (
        f"{column}_pct_change1"
    )
    rolling_mean_column = (
        f"{column}_rolling3_mean"
    )
    rolling_std_column = (
        f"{column}_rolling3_std"
    )

    # Previous calendar year's value
    feature_engineering_panel[
        lag_column
    ] = (
        feature_engineering_panel
        .groupby(
            history_group_columns,
            observed=True,
        )[column]
        .shift(1)
    )

    # Numerical difference from the previous year
    feature_engineering_panel[
        change_column
    ] = (
        feature_engineering_panel[column]
        - feature_engineering_panel[
            lag_column
        ]
    )

    # Percentage change is only meaningful where the
    # previous value exists and is not zero.
    valid_percentage_mask = (
        feature_engineering_panel[
            column
        ].notna()
        & feature_engineering_panel[
            lag_column
        ].notna()
        & feature_engineering_panel[
            lag_column
        ].ne(0)
    )

    feature_engineering_panel[
        percentage_column
    ] = np.nan

    feature_engineering_panel.loc[
        valid_percentage_mask,
        percentage_column,
    ] = (
        100
        * feature_engineering_panel.loc[
            valid_percentage_mask,
            change_column,
        ]
        / feature_engineering_panel.loc[
            valid_percentage_mask,
            lag_column,
        ].abs()
    )

    # Recent average and variability using the current
    # year and up to two preceding calendar years.
    feature_engineering_panel[
        rolling_mean_column
    ] = (
        feature_engineering_panel
        .groupby(
            history_group_columns,
            observed=True,
        )[column]
        .transform(
            lambda series: (
                series
                .rolling(
                    window=3,
                    min_periods=2,
                )
                .mean()
            )
        )
    )

    feature_engineering_panel[
        rolling_std_column
    ] = (
        feature_engineering_panel
        .groupby(
            history_group_columns,
            observed=True,
        )[column]
        .transform(
            lambda series: (
                series
                .rolling(
                    window=3,
                    min_periods=2,
                )
                .std(ddof=0)
            )
        )
    )

    lag1_columns.append(lag_column)
    change1_columns.append(change_column)
    percentage_change1_columns.append(
        percentage_column
    )
    rolling3_mean_columns.append(
        rolling_mean_column
    )
    rolling3_std_columns.append(
        rolling_std_column
    )

historical_feature_columns = (
    lag1_columns
    + change1_columns
    + percentage_change1_columns
    + rolling3_mean_columns
    + rolling3_std_columns
)

# ---------------------------------------------------------
# Create population-history features
# ---------------------------------------------------------

feature_engineering_panel[
    "Population_lag1"
] = (
    feature_engineering_panel
    .groupby(
        history_group_columns,
        observed=True,
    )["Population"]
    .shift(1)
)

valid_population_growth_mask = (
    feature_engineering_panel[
        "Population"
    ].notna()
    & feature_engineering_panel[
        "Population_lag1"
    ].notna()
    & feature_engineering_panel[
        "Population_lag1"
    ].gt(0)
)

feature_engineering_panel[
    "Population_growth_pct1"
] = np.nan

feature_engineering_panel.loc[
    valid_population_growth_mask,
    "Population_growth_pct1",
] = (
    100
    * (
        feature_engineering_panel.loc[
            valid_population_growth_mask,
            "Population",
        ]
        - feature_engineering_panel.loc[
            valid_population_growth_mask,
            "Population_lag1",
        ]
    )
    / feature_engineering_panel.loc[
        valid_population_growth_mask,
        "Population_lag1",
    ]
)

population_history_features = [
    "Population_lag1",
    "Population_growth_pct1",
]

all_history_features = (
    historical_feature_columns
    + population_history_features
)

# ---------------------------------------------------------
# Attach the historical features to the modelling rows
# ---------------------------------------------------------

history_payload = feature_engineering_panel[
    panel_key_columns
    + all_history_features
].copy()

model_history_features = (
    model_current_features
    .merge(
        history_payload,
        on=panel_key_columns,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["Area", "Item Code", "Year"]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Summarise historical-feature availability
# ---------------------------------------------------------

history_feature_groups = {
    "Previous-year analytical values": (
        lag1_columns
    ),
    "One-year absolute changes": (
        change1_columns
    ),
    "One-year percentage changes": (
        percentage_change1_columns
    ),
    "Three-year averages": (
        rolling3_mean_columns
    ),
    "Three-year variability measures": (
        rolling3_std_columns
    ),
    "Population-history features": (
        population_history_features
    ),
}

history_availability_rows = []

for split_name in split_order:
    split_mask = (
        model_history_features[
            "Temporal split"
        ].eq(split_name)
    )

    for group_name, columns in (
        history_feature_groups.items()
    ):
        selected_values = (
            model_history_features.loc[
                split_mask,
                columns,
            ]
        )

        total_cells = selected_values.size
        available_cells = (
            selected_values
            .notna()
            .to_numpy()
            .sum()
        )

        history_availability_rows.append(
            {
                "Temporal split": split_name,
                "Historical feature group": (
                    group_name
                ),
                "Features": len(columns),
                "Possible cells": total_cells,
                "Available cells": available_cells,
                "Availability %": (
                    100
                    * available_cells
                    / total_cells
                ),
            }
        )

history_availability_summary = pd.DataFrame(
    history_availability_rows
)

history_availability_summary[
    "Temporal split"
] = pd.Categorical(
    history_availability_summary[
        "Temporal split"
    ],
    categories=split_order,
    ordered=True,
)

history_availability_summary = (
    history_availability_summary
    .sort_values(
        [
            "Historical feature group",
            "Temporal split",
        ]
    )
    .reset_index(drop=True)
)

print("Historical-feature availability:")
display(history_availability_summary)

# ---------------------------------------------------------
# Display a human-readable example
# ---------------------------------------------------------

example_history_columns = [
    "Area",
    "Item",
    "Year",
    "food_supply_kcal_cap_day",
    "food_supply_kcal_cap_day_lag1",
    "food_supply_kcal_cap_day_change1",
    "food_supply_kcal_cap_day_pct_change1",
    "food_supply_kcal_cap_day_rolling3_mean",
    "food_supply_kcal_cap_day_rolling3_std",
    "Population",
    "Population_lag1",
    "Population_growth_pct1",
    target_column,
]

print(
    "\nExample: Algeria — Wheat and products:"
)

display(
    model_history_features.loc[
        model_history_features["Area"]
        .eq("Algeria")
        & model_history_features["Item"]
        .eq("Wheat and products"),
        example_history_columns,
    ].head(6)
)

# ---------------------------------------------------------
# Summarise the expanded table
# ---------------------------------------------------------

historical_table_summary = pd.Series(
    {
        "Rows": len(model_history_features),
        "Columns": (
            model_history_features.shape[1]
        ),
        "Historical analytical features": (
            len(historical_feature_columns)
        ),
        "Previous-year features": (
            len(lag1_columns)
        ),
        "Absolute-change features": (
            len(change1_columns)
        ),
        "Percentage-change features": (
            len(percentage_change1_columns)
        ),
        "Three-year average features": (
            len(rolling3_mean_columns)
        ),
        "Three-year variability features": (
            len(rolling3_std_columns)
        ),
        "Population-history features": (
            len(population_history_features)
        ),
        "Infinite historical values": (
            np.isinf(
                model_history_features[
                    all_history_features
                ].select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).sum()
        ),
        "Duplicate modelling keys": (
            model_history_features[
                panel_key_columns
            ].duplicated().sum()
        ),
    },
    name="Result",
).to_frame()

print("\nHistorical modelling-table summary:")
display(historical_table_summary)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(historical_feature_columns) == 70
assert len(population_history_features) == 2
assert len(all_history_features) == 72

assert len(model_history_features) == 3248
assert model_history_features.shape[1] == 142

assert (
    model_history_features[
        panel_key_columns
    ].duplicated().sum()
    == 0
)

assert model_history_features[
    target_column
].sum() == 334

# Recalculate each previous-year column independently
# and confirm it matches.
for column, lag_column in zip(
    historical_base_features,
    lag1_columns,
):
    expected_lag = (
        feature_engineering_panel
        .groupby(
            history_group_columns,
            observed=True,
        )[column]
        .shift(1)
    )

    assert np.allclose(
        feature_engineering_panel[
            lag_column
        ].to_numpy(dtype=float),
        expected_lag.to_numpy(dtype=float),
        equal_nan=True,
    )

# The first year of every complete country-commodity
# history must not have a previous-year value.
first_year_mask = (
    feature_engineering_panel["Year"]
    .eq(2010)
)

assert feature_engineering_panel.loc[
    first_year_mask,
    lag1_columns
    + population_history_features,
].isna().all().all()

# No infinite values are allowed.
assert not np.isinf(
    model_history_features[
        all_history_features
    ].select_dtypes(
        include=[np.number]
    ).to_numpy()
).any()

# Confirm that no future-oriented name entered the features.
for column in all_history_features:
    lowered_column = column.lower()

    assert "target" not in lowered_column
    assert "shortage" not in lowered_column
    assert "next_year" not in lowered_column
    assert "lead" not in lowered_column

print(
    "\nLeakage-safe historical features were "
    "created and validated successfully."
)

Historical-feature availability:


,Temporal split,Historical feature group,Features,Possible cells,Available cells,Availability %
0,Train,One-year absolute changes,14,31500,25845,82.05
1,Validation,One-year absolute changes,14,7000,6457,92.24
2,Test,One-year absolute changes,14,6972,6471,92.81
3,Train,One-year percentage changes,14,31500,20911,66.38
4,Validation,One-year percentage changes,14,7000,5254,75.06
5,Test,One-year percentage changes,14,6972,5277,75.69
6,Train,Population-history features,2,4500,4002,88.93
7,Validation,Population-history features,2,1000,1000,100.00
8,Test,Population-history features,2,996,996,100.00
9,Train,Previous-year analytical values,14,31500,25992,82.51



Example: Algeria — Wheat and products:


,Area,Item,Year,food_supply_kcal_cap_day,food_supply_kcal_cap_day_lag1,food_supply_kcal_cap_day_change1,food_supply_kcal_cap_day_pct_change1,food_supply_kcal_cap_day_rolling3_mean,food_supply_kcal_cap_day_rolling3_std,Population,Population_lag1,Population_growth_pct1,shortage_next_year
0,Algeria,Wheat and products,2010,"1,314.96",NaN,NaN,NaN,NaN,NaN,"36,188,240.00",NaN,NaN,0
1,Algeria,Wheat and products,2011,"1,287.02","1,314.96",-27.94,-2.12,"1,300.99",13.97,"36,903,380.00","36,188,240.00",1.98,0
2,Algeria,Wheat and products,2012,"1,312.79","1,287.02",25.77,2.00,"1,304.92",12.69,"37,646,170.00","36,903,380.00",2.01,0
3,Algeria,Wheat and products,2013,"1,270.79","1,312.79",-42.00,-3.20,"1,290.20",17.29,"38,414,170.00","37,646,170.00",2.04,0
4,Algeria,Wheat and products,2014,"1,252.06","1,270.79",-18.73,-1.47,"1,278.55",25.39,"39,205,030.00","38,414,170.00",2.06,0
5,Algeria,Wheat and products,2015,"1,259.57","1,252.06",7.51,0.60,"1,260.81",7.70,"40,019,530.00","39,205,030.00",2.08,0



Historical modelling-table summary:


,Result
Rows,3248
Columns,142
Historical analytical features,70
Previous-year features,14
Absolute-change features,14
Percentage-change features,14
Three-year average features,14
Three-year variability features,14
Population-history features,2
Infinite historical values,0



Leakage-safe historical features were created and validated successfully.


In [9]:
# ---------------------------------------------------------
# Helper functions for safe ratio calculations
# ---------------------------------------------------------

def safe_percentage_ratio(
    numerator,
    denominator,
):
    """
    Calculate numerator as a percentage of denominator.

    The result is left missing where either value is missing
    or where the denominator is zero or negative.
    """
    valid_mask = (
        numerator.notna()
        & denominator.notna()
        & denominator.gt(0)
    )

    result = pd.Series(
        np.nan,
        index=numerator.index,
        dtype="float64",
    )

    result.loc[valid_mask] = (
        100
        * numerator.loc[valid_mask]
        / denominator.loc[valid_mask]
    )

    return result


def thousand_tonnes_to_kg_per_person(
    quantity_1000t,
    population,
):
    """
    Convert a quantity measured in 1,000 tonnes
    into kilograms per person.

    One unit of 1,000 tonnes equals 1,000,000 kg.
    """
    valid_mask = (
        quantity_1000t.notna()
        & population.notna()
        & population.gt(0)
    )

    result = pd.Series(
        np.nan,
        index=quantity_1000t.index,
        dtype="float64",
    )

    result.loc[valid_mask] = (
        quantity_1000t.loc[valid_mask]
        * 1_000_000
        / population.loc[valid_mask]
    )

    return result


# ---------------------------------------------------------
# Create supply-pressure ratios
# ---------------------------------------------------------

domestic_supply = (
    feature_engineering_panel[
        "domestic_supply_1000t"
    ]
)

feature_engineering_panel[
    "import_share_of_domestic_supply_pct"
] = safe_percentage_ratio(
    feature_engineering_panel[
        "import_quantity_1000t"
    ],
    domestic_supply,
)

feature_engineering_panel[
    "production_share_of_domestic_supply_pct"
] = safe_percentage_ratio(
    feature_engineering_panel[
        "production_1000t"
    ],
    domestic_supply,
)

feature_engineering_panel[
    "export_share_of_domestic_supply_pct"
] = safe_percentage_ratio(
    feature_engineering_panel[
        "export_quantity_1000t"
    ],
    domestic_supply,
)

feature_engineering_panel[
    "food_share_of_domestic_supply_pct"
] = safe_percentage_ratio(
    feature_engineering_panel[
        "food_1000t"
    ],
    domestic_supply,
)

feature_engineering_panel[
    "losses_share_of_domestic_supply_pct"
] = safe_percentage_ratio(
    feature_engineering_panel[
        "losses_1000t"
    ],
    domestic_supply,
)

feature_engineering_panel[
    "stock_variation_share_of_domestic_supply_pct"
] = safe_percentage_ratio(
    feature_engineering_panel[
        "stock_variation_1000t"
    ],
    domestic_supply,
)

# ---------------------------------------------------------
# Create country-size-adjusted quantities
# ---------------------------------------------------------

feature_engineering_panel[
    "production_kg_per_person"
] = thousand_tonnes_to_kg_per_person(
    feature_engineering_panel[
        "production_1000t"
    ],
    feature_engineering_panel[
        "Population"
    ],
)

feature_engineering_panel[
    "imports_kg_per_person"
] = thousand_tonnes_to_kg_per_person(
    feature_engineering_panel[
        "import_quantity_1000t"
    ],
    feature_engineering_panel[
        "Population"
    ],
)

feature_engineering_panel[
    "exports_kg_per_person"
] = thousand_tonnes_to_kg_per_person(
    feature_engineering_panel[
        "export_quantity_1000t"
    ],
    feature_engineering_panel[
        "Population"
    ],
)

# Population is highly different in scale across countries.
# The logarithm compresses that scale while retaining order.
feature_engineering_panel[
    "log_population"
] = np.log1p(
    feature_engineering_panel[
        "Population"
    ]
)

derived_current_feature_columns = [
    "import_share_of_domestic_supply_pct",
    "production_share_of_domestic_supply_pct",
    "export_share_of_domestic_supply_pct",
    "food_share_of_domestic_supply_pct",
    "losses_share_of_domestic_supply_pct",
    "stock_variation_share_of_domestic_supply_pct",
    "production_kg_per_person",
    "imports_kg_per_person",
    "exports_kg_per_person",
    "log_population",
]

# ---------------------------------------------------------
# Attach the derived indicators to the modelling table
# ---------------------------------------------------------

derived_feature_payload = (
    feature_engineering_panel[
        panel_key_columns
        + derived_current_feature_columns
    ]
    .copy()
)

model_derived_features = (
    model_history_features
    .merge(
        derived_feature_payload,
        on=panel_key_columns,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["Area", "Item Code", "Year"]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Audit availability by temporal split
# ---------------------------------------------------------

derived_availability_rows = []

for split_name in split_order:
    split_mask = (
        model_derived_features[
            "Temporal split"
        ].eq(split_name)
    )

    split_values = model_derived_features.loc[
        split_mask,
        derived_current_feature_columns,
    ]

    possible_cells = split_values.size
    available_cells = (
        split_values
        .notna()
        .to_numpy()
        .sum()
    )

    derived_availability_rows.append(
        {
            "Temporal split": split_name,
            "Rows": split_mask.sum(),
            "Derived features": (
                len(
                    derived_current_feature_columns
                )
            ),
            "Possible cells": possible_cells,
            "Available cells": available_cells,
            "Availability %": (
                100
                * available_cells
                / possible_cells
            ),
        }
    )

derived_availability_summary = pd.DataFrame(
    derived_availability_rows
)

print("Derived-feature availability:")
display(derived_availability_summary)

# ---------------------------------------------------------
# Inspect training-period distributions
# ---------------------------------------------------------

training_derived_distribution = (
    model_derived_features.loc[
        model_derived_features[
            "Temporal split"
        ].eq("Train"),
        derived_current_feature_columns,
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
        ]
    )
    .T[
        [
            "count",
            "min",
            "1%",
            "5%",
            "50%",
            "95%",
            "99%",
            "max",
        ]
    ]
    .reset_index()
    .rename(
        columns={
            "index": "Derived feature"
        }
    )
)

print("\nTraining-period derived-feature distribution:")
display(training_derived_distribution)

# ---------------------------------------------------------
# Display an interpretable example
# ---------------------------------------------------------

print(
    "\nExample derived indicators — "
    "Algeria, Wheat and products:"
)

display(
    model_derived_features.loc[
        model_derived_features["Area"]
        .eq("Algeria")
        & model_derived_features["Item"]
        .eq("Wheat and products"),
        [
            "Area",
            "Item",
            "Year",
            "production_1000t",
            "import_quantity_1000t",
            "domestic_supply_1000t",
            "Population",
            "production_share_of_domestic_supply_pct",
            "import_share_of_domestic_supply_pct",
            "production_kg_per_person",
            "imports_kg_per_person",
            target_column,
        ],
    ].head(5)
)

# ---------------------------------------------------------
# Summarise the expanded table
# ---------------------------------------------------------

derived_table_summary = pd.Series(
    {
        "Rows": len(model_derived_features),
        "Columns": (
            model_derived_features.shape[1]
        ),
        "New derived current-year features": (
            len(
                derived_current_feature_columns
            )
        ),
        "Missing derived-feature cells": (
            model_derived_features[
                derived_current_feature_columns
            ].isna().sum().sum()
        ),
        "Infinite derived-feature cells": (
            np.isinf(
                model_derived_features[
                    derived_current_feature_columns
                ].to_numpy(dtype=float)
            ).sum()
        ),
        "Duplicate modelling keys": (
            model_derived_features[
                panel_key_columns
            ].duplicated().sum()
        ),
    },
    name="Result",
).to_frame()

print("\nDerived-feature modelling-table summary:")
display(derived_table_summary)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    derived_current_feature_columns
) == 10

assert len(model_derived_features) == 3248
assert model_derived_features.shape[1] == 152

assert (
    model_derived_features[
        panel_key_columns
    ].duplicated().sum()
    == 0
)

assert model_derived_features[
    target_column
].sum() == 334

assert not np.isinf(
    model_derived_features[
        derived_current_feature_columns
    ].to_numpy(dtype=float)
).any()

assert model_derived_features[
    "log_population"
].notna().all()

# Confirm that all per-person conversions are based
# only on positive population values.
assert model_derived_features[
    "Population"
].gt(0).all()

for column in derived_current_feature_columns:
    lowered_column = column.lower()

    assert "target" not in lowered_column
    assert "shortage" not in lowered_column
    assert "next_year" not in lowered_column
    assert "lead" not in lowered_column

print(
    "\nSupply-pressure and country-size-adjusted "
    "features were created and validated successfully."
)

Derived-feature availability:


/var/folders/0_/w34bfn0s031cp4c5lggmkck80000gn/T/ipykernel_13669/414975980.py:157: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_engineering_panel[
/var/folders/0_/w34bfn0s031cp4c5lggmkck80000gn/T/ipykernel_13669/414975980.py:170: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_engineering_panel[


,Temporal split,Rows,Derived features,Possible cells,Available cells,Availability %
0,Train,2250,10,22500,20361,90.49
1,Validation,500,10,5000,4525,90.50
2,Test,498,10,4980,4551,91.39



Training-period derived-feature distribution:


,Derived feature,count,min,1%,5%,50%,95%,99%,max
0,import_share_of_domestic_supply_pct,"2,096.00",0.00,0.00,0.00,14.80,113.53,161.71,"2,500.00"
1,production_share_of_domestic_supply_pct,"1,979.00",0.00,0.00,0.23,95.42,116.01,144.84,200.00
2,export_share_of_domestic_supply_pct,"1,774.00",0.00,0.00,0.00,0.00,20.97,47.47,609.43
3,food_share_of_domestic_supply_pct,"2,235.00",0.00,4.23,21.59,80.66,100.00,100.00,600.94
4,losses_share_of_domestic_supply_pct,"2,024.00",0.00,0.00,0.00,4.72,16.67,24.11,35.86
5,stock_variation_share_of_domestic_supply_pct,"2,131.00",-91.63,-39.40,-16.72,0.00,25.65,64.19,"2,400.00"
6,production_kg_per_person,"1,987.00",0.00,0.00,0.05,10.34,224.86,523.37,680.41
7,imports_kg_per_person,"2,111.00",0.00,0.00,0.00,1.60,120.73,284.00,796.82
8,exports_kg_per_person,"1,774.00",0.00,0.00,0.00,0.00,5.70,27.27,61.30
9,log_population,"2,250.00",11.46,11.56,13.15,16.51,18.43,19.09,19.14



Example derived indicators — Algeria, Wheat and products:


,Area,Item,Year,production_1000t,import_quantity_1000t,domestic_supply_1000t,Population,production_share_of_domestic_supply_pct,import_share_of_domestic_supply_pct,production_kg_per_person,imports_kg_per_person,shortage_next_year
0,Algeria,Wheat and products,2010,"2,605.00","5,109.00","8,943.00","36,188,240.00",29.13,57.13,71.98,141.18,0
1,Algeria,Wheat and products,2011,"2,911.00","7,487.00","9,747.00","36,903,380.00",29.87,76.81,78.88,202.88,0
2,Algeria,Wheat and products,2012,"3,432.00","6,390.00","9,868.00","37,646,170.00",34.78,64.75,91.16,169.74,0
3,Algeria,Wheat and products,2013,"3,299.00","6,344.00","9,722.00","38,414,170.00",33.93,65.25,85.88,165.15,0
4,Algeria,Wheat and products,2014,"2,436.00","7,481.00","9,413.00","39,205,030.00",25.88,79.48,62.13,190.82,0



Derived-feature modelling-table summary:


,Result
Rows,3248
Columns,152
New derived current-year features,10
Missing derived-feature cells,3043
Infinite derived-feature cells,0
Duplicate modelling keys,0



Supply-pressure and country-size-adjusted features were created and validated successfully.


In [10]:
# ---------------------------------------------------------
# Defragment the DataFrames after repeated column creation
# ---------------------------------------------------------

feature_engineering_panel = (
    feature_engineering_panel.copy()
)

model_derived_features = (
    model_derived_features.copy()
)

# ---------------------------------------------------------
# Define candidate predictors by modelling type
# ---------------------------------------------------------

categorical_feature_columns = [
    "Area",
    "Item Code",
    "production_status",
]

binary_indicator_feature_columns = (
    source_indicator_features
    + missing_indicator_features
)

numeric_continuous_feature_columns = (
    ["Year"]
    + population_features
    + current_analytical_features
    + numeric_quality_features
    + historical_feature_columns
    + population_history_features
    + derived_current_feature_columns
)

candidate_feature_columns = (
    categorical_feature_columns
    + binary_indicator_feature_columns
    + numeric_continuous_feature_columns
)

# Confirm that no feature name was accidentally repeated
duplicated_candidate_names = (
    pd.Series(candidate_feature_columns)
    .duplicated()
)

if duplicated_candidate_names.any():
    duplicated_names = (
        pd.Series(candidate_feature_columns)[
            duplicated_candidate_names
        ]
        .tolist()
    )

    raise ValueError(
        "Duplicated candidate feature names found:\n"
        + "\n".join(
            f"- {column}"
            for column in duplicated_names
        )
    )

# ---------------------------------------------------------
# Screen features using training data only
# ---------------------------------------------------------

training_feature_data = (
    model_derived_features.loc[
        model_derived_features[
            "Temporal split"
        ].eq("Train"),
        candidate_feature_columns,
    ]
)

all_missing_training_features = [
    column
    for column in candidate_feature_columns
    if training_feature_data[column]
    .isna()
    .all()
]

constant_training_features = [
    column
    for column in candidate_feature_columns
    if training_feature_data[column]
    .nunique(dropna=False)
    <= 1
]

removed_training_features = sorted(
    set(
        all_missing_training_features
        + constant_training_features
    )
)

final_feature_columns = [
    column
    for column in candidate_feature_columns
    if column not in removed_training_features
]

final_categorical_features = [
    column
    for column in categorical_feature_columns
    if column in final_feature_columns
]

final_binary_indicator_features = [
    column
    for column in binary_indicator_feature_columns
    if column in final_feature_columns
]

final_numeric_continuous_features = [
    column
    for column in numeric_continuous_feature_columns
    if column in final_feature_columns
]

# ---------------------------------------------------------
# Create a feature registry
# ---------------------------------------------------------

feature_registry_rows = []

for column in candidate_feature_columns:
    if column in categorical_feature_columns:
        feature_type = "Categorical"

    elif column in binary_indicator_feature_columns:
        feature_type = "Binary indicator"

    else:
        feature_type = "Numeric"

    training_missing_count = (
        training_feature_data[column]
        .isna()
        .sum()
    )

    training_missing_pct = (
        100
        * training_missing_count
        / len(training_feature_data)
    )

    if column in all_missing_training_features:
        status = "Removed"
        reason = (
            "Entirely missing in the training period"
        )

    elif column in constant_training_features:
        status = "Removed"
        reason = (
            "Contains no variation in the "
            "training period"
        )

    else:
        status = "Retained"
        reason = (
            "Contains usable training-period variation"
        )

    feature_registry_rows.append(
        {
            "Feature": column,
            "Feature type": feature_type,
            "Training missing values": (
                training_missing_count
            ),
            "Training missing %": (
                training_missing_pct
            ),
            "Training unique values including missing": (
                training_feature_data[column]
                .nunique(dropna=False)
            ),
            "Status": status,
            "Reason": reason,
        }
    )

feature_registry = pd.DataFrame(
    feature_registry_rows
)

# ---------------------------------------------------------
# Summarise the screening results
# ---------------------------------------------------------

feature_screening_summary = pd.Series(
    {
        "Candidate features": (
            len(candidate_feature_columns)
        ),
        "Candidate categorical features": (
            len(categorical_feature_columns)
        ),
        "Candidate binary indicators": (
            len(binary_indicator_feature_columns)
        ),
        "Candidate numeric features": (
            len(numeric_continuous_feature_columns)
        ),
        "Entirely missing training features": (
            len(all_missing_training_features)
        ),
        "Constant training features": (
            len(constant_training_features)
        ),
        "Unique features removed": (
            len(removed_training_features)
        ),
        "Final retained features": (
            len(final_feature_columns)
        ),
        "Final categorical features": (
            len(final_categorical_features)
        ),
        "Final binary indicators": (
            len(final_binary_indicator_features)
        ),
        "Final numeric features": (
            len(final_numeric_continuous_features)
        ),
    },
    name="Result",
).to_frame()

print("Training-only feature screening summary:")
display(feature_screening_summary)

print("\nFeatures removed because they contain no training variation:")

if removed_training_features:
    display(
        feature_registry.loc[
            feature_registry["Feature"]
            .isin(removed_training_features)
        ].reset_index(drop=True)
    )
else:
    print("None")

# Show retained features with substantial missingness
high_missing_retained_features = (
    feature_registry.loc[
        feature_registry["Status"].eq("Retained")
        & feature_registry[
            "Training missing %"
        ].ge(50)
    ]
    .sort_values(
        "Training missing %",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\nRetained features with at least "
    "50% missing training values:"
)

if len(high_missing_retained_features):
    display(high_missing_retained_features)
else:
    print("None")

# ---------------------------------------------------------
# Record the treatment policy for later modelling
# ---------------------------------------------------------

preprocessing_policy = pd.DataFrame(
    {
        "Feature type": [
            "Categorical features",
            "Binary indicators",
            "Ordinary numeric features",
            "Numeric features with missing values",
            "Extreme numeric values",
        ],
        "Planned treatment": [
            "One-hot encoding learned from training data",
            "Retained as 0/1 values",
            "Scaling learned from training data where required",
            "Imputation learned from training data, with missingness preserved",
            "Training-derived lower and upper boundaries; raw saved values remain unchanged",
        ],
        "Why": [
            "Models require numerical representations of names and categories",
            "Their existing values already have a direct interpretation",
            "Prevents large-unit variables from dominating scale-sensitive models",
            "Validation and test information must not determine replacement values",
            "Prevents a few unusually large ratios from dominating the model",
        ],
    }
)

print("\nPlanned modelling-preprocessing policy:")
display(preprocessing_policy)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(candidate_feature_columns) == 144

assert len(
    set(candidate_feature_columns)
) == len(candidate_feature_columns)

assert target_column not in candidate_feature_columns
assert "target_year" not in candidate_feature_columns
assert "Temporal split" not in candidate_feature_columns
assert "Item" not in candidate_feature_columns

assert set(final_feature_columns).isdisjoint(
    removed_training_features
)

assert (
    len(final_categorical_features)
    + len(final_binary_indicator_features)
    + len(final_numeric_continuous_features)
    == len(final_feature_columns)
)

# Every retained training feature must contain at least
# two distinct states when missingness is included.
for column in final_feature_columns:
    assert (
        training_feature_data[column]
        .nunique(dropna=False)
        >= 2
    )

# None of the feature names may imply future information.
for column in final_feature_columns:
    lowered_column = column.lower()

    assert "target" not in lowered_column
    assert "shortage" not in lowered_column
    assert "next_year" not in lowered_column
    assert "lead" not in lowered_column

print(
    "\nThe final modelling feature registry was "
    "created using training data only."
)

Training-only feature screening summary:


,Result
Candidate features,144
Candidate categorical features,3
Candidate binary indicators,34
Candidate numeric features,107
Entirely missing training features,0
Constant training features,14
Unique features removed,14
Final retained features,130
Final categorical features,3
Final binary indicators,20



Features removed because they contain no training variation:


,Feature,Feature type,Training missing values,Training missing %,Training unique values including missing,Status,Reason
0,domestic_supply_1000t_source_present,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
1,fat_supply_g_cap_day_source_present,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
2,food_1000t_source_present,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
3,food_supply_kcal_cap_day_source_present,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
4,food_supply_quantity_kg_cap_yr_source_present,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
5,protein_supply_g_cap_day_source_present,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
6,residuals_1000t_source_present,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
7,domestic_supply_1000t_missing,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
8,fat_supply_g_cap_day_missing,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period
9,food_1000t_missing,Binary indicator,0,0.00,1,Removed,Contains no variation in the training period



Retained features with at least 50% missing training values:


,Feature,Feature type,Training missing values,Training missing %,Training unique values including missing,Status,Reason
0,residuals_1000t_pct_change1,Numeric,2062,91.64,96,Retained,Contains usable training-period variation
1,tourist_consumption_1000t,Numeric,1868,83.02,22,Retained,Contains usable training-period variation
2,export_quantity_1000t_pct_change1,Numeric,1470,65.33,363,Retained,Contains usable training-period variation
3,other_uses_non_food_1000t,Numeric,1428,63.47,185,Retained,Contains usable training-period variation
4,seed_1000t_pct_change1,Numeric,1143,50.80,372,Retained,Contains usable training-period variation
5,feed_1000t_pct_change1,Numeric,1132,50.31,686,Retained,Contains usable training-period variation



Planned modelling-preprocessing policy:


,Feature type,Planned treatment,Why
0,Categorical features,One-hot encoding learned from training data,Models require numerical representations of na...
1,Binary indicators,Retained as 0/1 values,Their existing values already have a direct in...
2,Ordinary numeric features,Scaling learned from training data where required,Prevents large-unit variables from dominating ...
3,Numeric features with missing values,"Imputation learned from training data, with mi...",Validation and test information must not deter...
4,Extreme numeric values,Training-derived lower and upper boundaries; r...,Prevents a few unusually large ratios from dom...



The final modelling feature registry was created using training data only.


In [11]:
# ---------------------------------------------------------
# Remove one historically unstable percentage feature
# ---------------------------------------------------------

manual_feature_exclusions = {
    "residuals_1000t_pct_change1": (
        "Removed because 91.64% of training values are "
        "missing and residuals are usually zero, making "
        "percentage change undefined and unstable"
    )
}

for column, reason in (
    manual_feature_exclusions.items()
):
    if column in final_feature_columns:
        final_feature_columns.remove(column)

    if column in final_numeric_continuous_features:
        final_numeric_continuous_features.remove(column)

    feature_registry.loc[
        feature_registry["Feature"].eq(column),
        "Status",
    ] = "Removed after suitability review"

    feature_registry.loc[
        feature_registry["Feature"].eq(column),
        "Reason",
    ] = reason

# ---------------------------------------------------------
# Create a stable identifier for every modelling observation
# ---------------------------------------------------------

model_dataset_source = (
    model_derived_features.copy()
)

model_dataset_source[
    "observation_id"
] = (
    model_dataset_source["M49 Code"]
    .astype(str)
    .str.zfill(3)
    + "_"
    + model_dataset_source["Item Code"]
    .astype(str)
    + "_"
    + model_dataset_source["Year"]
    .astype(str)
)

non_feature_metadata_columns = [
    "observation_id",
    "Area Code",
    "Area Code (M49)",
    "M49 Code",
    "Item Code (FBS)",
    "Item",
    "target_year",
    "Temporal split",
    target_column,
]

modelling_dataset_columns = list(
    dict.fromkeys(
        non_feature_metadata_columns
        + final_feature_columns
    )
)

modelling_dataset = (
    model_dataset_source[
        modelling_dataset_columns
    ]
    .copy()
    .sort_values(
        ["Area", "Item Code", "Year"]
    )
    .reset_index(drop=True)
)

# Make the intended modelling roles explicit.
for column in final_categorical_features:
    modelling_dataset[column] = (
        modelling_dataset[column]
        .astype(str)
    )

for column in final_binary_indicator_features:
    modelling_dataset[column] = (
        modelling_dataset[column]
        .astype("int8")
    )

for column in final_numeric_continuous_features:
    modelling_dataset[column] = (
        pd.to_numeric(
            modelling_dataset[column],
            errors="coerce",
        )
    )

# ---------------------------------------------------------
# Create the chronological modelling matrices
# ---------------------------------------------------------

train_mask = (
    modelling_dataset[
        "Temporal split"
    ].eq("Train")
)

validation_mask = (
    modelling_dataset[
        "Temporal split"
    ].eq("Validation")
)

test_mask = (
    modelling_dataset[
        "Temporal split"
    ].eq("Test")
)

X_train = modelling_dataset.loc[
    train_mask,
    final_feature_columns,
].copy()

y_train = modelling_dataset.loc[
    train_mask,
    target_column,
].astype("int8").copy()

X_validation = modelling_dataset.loc[
    validation_mask,
    final_feature_columns,
].copy()

y_validation = modelling_dataset.loc[
    validation_mask,
    target_column,
].astype("int8").copy()

X_test = modelling_dataset.loc[
    test_mask,
    final_feature_columns,
].copy()

y_test = modelling_dataset.loc[
    test_mask,
    target_column,
].astype("int8").copy()

# Separate metadata is retained for later interpretation.
train_metadata = modelling_dataset.loc[
    train_mask,
    non_feature_metadata_columns,
].copy()

validation_metadata = modelling_dataset.loc[
    validation_mask,
    non_feature_metadata_columns,
].copy()

test_metadata = modelling_dataset.loc[
    test_mask,
    non_feature_metadata_columns,
].copy()

# ---------------------------------------------------------
# Summarise the three chronological matrices
# ---------------------------------------------------------

split_matrix_rows = []

for split_name, split_mask, X_split, y_split in [
    (
        "Train",
        train_mask,
        X_train,
        y_train,
    ),
    (
        "Validation",
        validation_mask,
        X_validation,
        y_validation,
    ),
    (
        "Test",
        test_mask,
        X_test,
        y_test,
    ),
]:
    split_rows = modelling_dataset.loc[
        split_mask
    ]

    split_matrix_rows.append(
        {
            "Temporal split": split_name,
            "Rows": len(X_split),
            "Predictors": X_split.shape[1],
            "Predictor year start": (
                split_rows["Year"].min()
            ),
            "Predictor year end": (
                split_rows["Year"].max()
            ),
            "Target year start": (
                split_rows["target_year"].min()
            ),
            "Target year end": (
                split_rows["target_year"].max()
            ),
            "Positive events": y_split.sum(),
            "Negative outcomes": (
                len(y_split) - y_split.sum()
            ),
            "Event rate %": (
                100 * y_split.mean()
            ),
            "Missing predictor cells": (
                X_split.isna().sum().sum()
            ),
            "Missing predictor %": (
                100
                * X_split.isna().sum().sum()
                / X_split.size
            ),
        }
    )

split_matrix_summary = pd.DataFrame(
    split_matrix_rows
)

print("Chronological modelling-matrix summary:")
display(split_matrix_summary)

# ---------------------------------------------------------
# Check whether later periods contain unseen categories
# ---------------------------------------------------------

category_check_rows = []

for column in final_categorical_features:
    training_categories = set(
        X_train[column]
        .dropna()
        .astype(str)
        .unique()
    )

    validation_categories = set(
        X_validation[column]
        .dropna()
        .astype(str)
        .unique()
    )

    test_categories = set(
        X_test[column]
        .dropna()
        .astype(str)
        .unique()
    )

    unseen_validation = sorted(
        validation_categories
        - training_categories
    )

    unseen_test = sorted(
        test_categories
        - training_categories
    )

    category_check_rows.append(
        {
            "Categorical feature": column,
            "Training categories": (
                len(training_categories)
            ),
            "Validation categories": (
                len(validation_categories)
            ),
            "Test categories": (
                len(test_categories)
            ),
            "Unseen validation categories": (
                len(unseen_validation)
            ),
            "Unseen test categories": (
                len(unseen_test)
            ),
            "Validation examples": (
                ", ".join(
                    unseen_validation[:5]
                )
                if unseen_validation
                else "None"
            ),
            "Test examples": (
                ", ".join(
                    unseen_test[:5]
                )
                if unseen_test
                else "None"
            ),
        }
    )

category_check_summary = pd.DataFrame(
    category_check_rows
)

print("\nCategorical-level check:")
display(category_check_summary)

# ---------------------------------------------------------
# Summarise the final raw modelling dataset
# ---------------------------------------------------------

final_dataset_summary = pd.Series(
    {
        "Rows": len(modelling_dataset),
        "Columns including metadata and target": (
            modelling_dataset.shape[1]
        ),
        "Final predictors": (
            len(final_feature_columns)
        ),
        "Categorical predictors": (
            len(final_categorical_features)
        ),
        "Binary indicator predictors": (
            len(final_binary_indicator_features)
        ),
        "Numeric predictors": (
            len(final_numeric_continuous_features)
        ),
        "Predictor year start": (
            modelling_dataset["Year"].min()
        ),
        "Predictor year end": (
            modelling_dataset["Year"].max()
        ),
        "Target year start": (
            modelling_dataset["target_year"].min()
        ),
        "Target year end": (
            modelling_dataset["target_year"].max()
        ),
        "Positive events": (
            modelling_dataset[
                target_column
            ].sum()
        ),
        "Duplicate observation identifiers": (
            modelling_dataset[
                "observation_id"
            ].duplicated().sum()
        ),
        "Duplicate modelling keys": (
            modelling_dataset[
                panel_key_columns
            ].duplicated().sum()
        ),
    },
    name="Result",
).to_frame()

print("\nFinal raw modelling-dataset summary:")
display(final_dataset_summary)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(final_feature_columns) == 129
assert len(final_categorical_features) == 3
assert len(final_binary_indicator_features) == 20
assert len(final_numeric_continuous_features) == 106

assert X_train.shape == (2250, 129)
assert X_validation.shape == (500, 129)
assert X_test.shape == (498, 129)

assert y_train.sum() == 231
assert y_validation.sum() == 52
assert y_test.sum() == 51

assert set(X_train.index).isdisjoint(
    X_validation.index
)

assert set(X_train.index).isdisjoint(
    X_test.index
)

assert set(X_validation.index).isdisjoint(
    X_test.index
)

assert target_column not in X_train.columns
assert "target_year" not in X_train.columns
assert "Temporal split" not in X_train.columns

assert list(X_train.columns) == final_feature_columns
assert list(X_validation.columns) == final_feature_columns
assert list(X_test.columns) == final_feature_columns

assert modelling_dataset[
    "observation_id"
].is_unique

assert (
    modelling_dataset[
        panel_key_columns
    ].duplicated().sum()
    == 0
)

assert modelling_dataset[
    target_column
].notna().all()

assert not np.isinf(
    modelling_dataset[
        final_numeric_continuous_features
    ].to_numpy(dtype=float)
).any()

assert modelling_dataset.shape == (
    3248,
    138,
)

print(
    "\nThe train, validation and test matrices "
    "were created and validated successfully."
)

Chronological modelling-matrix summary:


,Temporal split,Rows,Predictors,Predictor year start,Predictor year end,Target year start,Target year end,Positive events,Negative outcomes,Event rate %,Missing predictor cells,Missing predictor %
0,Train,2250,129,2010,2018,2011,2019,231,2019,10.27,39985,13.78
1,Validation,500,129,2019,2020,2020,2021,52,448,10.40,5356,8.30
2,Test,498,129,2021,2022,2022,2023,51,447,10.24,5083,7.91



Categorical-level check:


,Categorical feature,Training categories,Validation categories,Test categories,Unseen validation categories,Unseen test categories,Validation examples,Test examples
0,Area,43,43,43,0,0,None,None
1,Item Code,8,8,8,0,0,None,None
2,production_status,4,4,4,0,0,None,None



Final raw modelling-dataset summary:


,Result
Rows,3248
Columns including metadata and target,138
Final predictors,129
Categorical predictors,3
Binary indicator predictors,20
Numeric predictors,106
Predictor year start,2010
Predictor year end,2022
Target year start,2011
Target year end,2023



The train, validation and test matrices were created and validated successfully.


In [12]:
import json

# ---------------------------------------------------------
# Define Notebook 3 output paths
# ---------------------------------------------------------

MODEL_DATASET_PATH = (
    PROCESSED_DIR
    / "africa_model_ready_shortage_features_2010_2022.parquet"
)

FEATURE_REGISTRY_PATH = (
    PROCESSED_DIR
    / "shortage_feature_registry.csv"
)

FEATURE_SPECIFICATION_PATH = (
    PROCESSED_DIR
    / "shortage_feature_specification.json"
)

MODELLING_SPLIT_SUMMARY_PATH = (
    PROCESSED_DIR
    / "shortage_modelling_split_summary.csv"
)

PREPROCESSING_POLICY_PATH = (
    PROCESSED_DIR
    / "shortage_preprocessing_policy.csv"
)

# ---------------------------------------------------------
# Create the machine-readable feature specification
# ---------------------------------------------------------

feature_specification = {
    "dataset_name": (
        "Africa next-year commodity shortage "
        "modelling dataset"
    ),
    "prediction_horizon": "One year ahead",
    "target_column": target_column,
    "target_definition": {
        "minimum_current_kcal_capita_day": 5,
        "minimum_percentage_decline": 15,
        "minimum_absolute_decline_kcal_capita_day": 5,
    },
    "panel_key_columns": panel_key_columns,
    "observation_identifier": "observation_id",
    "final_feature_count": len(
        final_feature_columns
    ),
    "final_feature_columns": (
        final_feature_columns
    ),
    "categorical_features": (
        final_categorical_features
    ),
    "binary_indicator_features": (
        final_binary_indicator_features
    ),
    "numeric_features": (
        final_numeric_continuous_features
    ),
    "historical_coverage_threshold_pct": (
        historical_coverage_threshold
    ),
    "historical_base_features": (
        historical_base_features
    ),
    "sparse_current_only_features": (
        sparse_current_only_features
    ),
    "removed_training_constant_features": (
        removed_training_features
    ),
    "manual_feature_exclusions": (
        manual_feature_exclusions
    ),
    "temporal_splits": {
        "Train": {
            "predictor_years": [2010, 2018],
            "target_years": [2011, 2019],
        },
        "Validation": {
            "predictor_years": [2019, 2020],
            "target_years": [2020, 2021],
        },
        "Test": {
            "predictor_years": [2021, 2022],
            "target_years": [2022, 2023],
        },
    },
    "future_information_used": False,
    "raw_missing_values_preserved": True,
    "raw_extreme_values_preserved": True,
}

# ---------------------------------------------------------
# Save all Notebook 3 outputs
# ---------------------------------------------------------

modelling_dataset.to_parquet(
    MODEL_DATASET_PATH,
    index=False,
)

feature_registry.to_csv(
    FEATURE_REGISTRY_PATH,
    index=False,
)

with open(
    FEATURE_SPECIFICATION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_specification,
        file,
        indent=2,
        ensure_ascii=False,
    )

split_matrix_summary.to_csv(
    MODELLING_SPLIT_SUMMARY_PATH,
    index=False,
)

preprocessing_policy.to_csv(
    PREPROCESSING_POLICY_PATH,
    index=False,
)

# ---------------------------------------------------------
# Reload the saved outputs
# ---------------------------------------------------------

reloaded_modelling_dataset = pd.read_parquet(
    MODEL_DATASET_PATH
)

reloaded_feature_registry = pd.read_csv(
    FEATURE_REGISTRY_PATH
)

with open(
    FEATURE_SPECIFICATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    reloaded_feature_specification = (
        json.load(file)
    )

reloaded_split_summary = pd.read_csv(
    MODELLING_SPLIT_SUMMARY_PATH
)

reloaded_preprocessing_policy = pd.read_csv(
    PREPROCESSING_POLICY_PATH
)

# ---------------------------------------------------------
# Validate the reloaded modelling dataset
# ---------------------------------------------------------

pd.testing.assert_frame_equal(
    reloaded_modelling_dataset,
    modelling_dataset,
    check_dtype=False,
    check_categorical=False,
)

assert reloaded_modelling_dataset.shape == (
    3248,
    138,
)

assert reloaded_modelling_dataset[
    target_column
].sum() == 334

assert reloaded_modelling_dataset[
    "observation_id"
].is_unique

assert (
    reloaded_modelling_dataset[
        panel_key_columns
    ].duplicated().sum()
    == 0
)

assert reloaded_modelling_dataset[
    target_column
].notna().all()

# ---------------------------------------------------------
# Validate the feature registry and specification
# ---------------------------------------------------------

assert len(
    reloaded_feature_registry
) == 144

assert (
    reloaded_feature_registry[
        "Status"
    ].eq("Retained").sum()
    == 129
)

assert (
    reloaded_feature_specification[
        "final_feature_count"
    ]
    == 129
)

assert (
    reloaded_feature_specification[
        "final_feature_columns"
    ]
    == final_feature_columns
)

assert (
    reloaded_feature_specification[
        "categorical_features"
    ]
    == final_categorical_features
)

assert (
    reloaded_feature_specification[
        "binary_indicator_features"
    ]
    == final_binary_indicator_features
)

assert (
    reloaded_feature_specification[
        "numeric_features"
    ]
    == final_numeric_continuous_features
)

assert (
    reloaded_feature_specification[
        "future_information_used"
    ]
    is False
)

# ---------------------------------------------------------
# Validate the saved split summary
# ---------------------------------------------------------

assert reloaded_split_summary[
    "Rows"
].tolist() == [
    2250,
    500,
    498,
]

assert reloaded_split_summary[
    "Positive events"
].tolist() == [
    231,
    52,
    51,
]

assert len(
    reloaded_preprocessing_policy
) == 5

# ---------------------------------------------------------
# Display saved-file information
# ---------------------------------------------------------

saved_output_rows = []

for path, description in [
    (
        MODEL_DATASET_PATH,
        "Raw model-ready feature dataset",
    ),
    (
        FEATURE_REGISTRY_PATH,
        "Feature inclusion and exclusion registry",
    ),
    (
        FEATURE_SPECIFICATION_PATH,
        "Machine-readable feature specification",
    ),
    (
        MODELLING_SPLIT_SUMMARY_PATH,
        "Chronological split summary",
    ),
    (
        PREPROCESSING_POLICY_PATH,
        "Planned preprocessing policy",
    ),
]:
    saved_output_rows.append(
        {
            "Output": description,
            "File": path.name,
            "File size MB": (
                path.stat().st_size
                / (1024 ** 2)
            ),
        }
    )

saved_output_summary = pd.DataFrame(
    saved_output_rows
)

print("Saved Notebook 3 outputs:")
display(saved_output_summary)

final_checkpoint_summary = pd.Series(
    {
        "Saved modelling rows": (
            len(
                reloaded_modelling_dataset
            )
        ),
        "Saved modelling columns": (
            reloaded_modelling_dataset
            .shape[1]
        ),
        "Saved predictors": (
            reloaded_feature_specification[
                "final_feature_count"
            ]
        ),
        "Training rows": (
            reloaded_modelling_dataset[
                "Temporal split"
            ].eq("Train").sum()
        ),
        "Validation rows": (
            reloaded_modelling_dataset[
                "Temporal split"
            ].eq("Validation").sum()
        ),
        "Test rows": (
            reloaded_modelling_dataset[
                "Temporal split"
            ].eq("Test").sum()
        ),
        "Positive events": (
            reloaded_modelling_dataset[
                target_column
            ].sum()
        ),
        "Missing target values": (
            reloaded_modelling_dataset[
                target_column
            ].isna().sum()
        ),
        "Duplicate observation identifiers": (
            reloaded_modelling_dataset[
                "observation_id"
            ].duplicated().sum()
        ),
        "Future information used": (
            reloaded_feature_specification[
                "future_information_used"
            ]
        ),
    },
    name="Result",
).to_frame()

print("\nNotebook 3 checkpoint summary:")
display(final_checkpoint_summary)

print("\nSaved files:")

for path in [
    MODEL_DATASET_PATH,
    FEATURE_REGISTRY_PATH,
    FEATURE_SPECIFICATION_PATH,
    MODELLING_SPLIT_SUMMARY_PATH,
    PREPROCESSING_POLICY_PATH,
]:
    print(
        f"{path.name}: "
        f"{path.stat().st_size / (1024 ** 2):.3f} MB"
    )
    print(f"  {path}")

print(
    "\nAll Notebook 3 outputs were saved, "
    "reloaded and validated successfully."
)

Saved Notebook 3 outputs:


,Output,File,File size MB
0,Raw model-ready feature dataset,africa_model_ready_shortage_features_2010_2022...,1.17
1,Feature inclusion and exclusion registry,shortage_feature_registry.csv,0.02
2,Machine-readable feature specification,shortage_feature_specification.json,0.01
3,Chronological split summary,shortage_modelling_split_summary.csv,0.00
4,Planned preprocessing policy,shortage_preprocessing_policy.csv,0.00



Notebook 3 checkpoint summary:


,Result
Saved modelling rows,3248
Saved modelling columns,138
Saved predictors,129
Training rows,2250
Validation rows,500
Test rows,498
Positive events,334
Missing target values,0
Duplicate observation identifiers,0
Future information used,False



Saved files:
africa_model_ready_shortage_features_2010_2022.parquet: 1.167 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/africa_model_ready_shortage_features_2010_2022.parquet
shortage_feature_registry.csv: 0.015 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/shortage_feature_registry.csv
shortage_feature_specification.json: 0.011 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/shortage_feature_specification.json
shortage_modelling_split_summary.csv: 0.000 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/shortage_modelling_split_summary.csv
shortage_preprocessing_policy.csv: 0.001 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/shortage_preprocessing_policy.csv

All Notebook 3 outputs were saved, reloaded and validated successfully.
